In [2]:
import pandas as pd
import numpy as np
import json
import os
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load dữ liệu đã qua Feature Engineering ở Phase 2
df_train = pd.read_csv('../data/processed/train_features.csv')
df_test = pd.read_csv('../data/processed/test_features.csv')

print(f"Loaded Train Shape: {df_train.shape}")
print(f"Loaded Test Shape: {df_test.shape}")

Loaded Train Shape: (10350, 49)
Loaded Test Shape: (3675, 49)


In [6]:
# Danh sách các cột phi số học, biến định danh và biến target
metadata_cols = ['date', 'country_name', 'region', 'income_level', 'vector_disease_risk_score', 'target_log', ]

# 🚨 BIỆN PHÁP CHỐNG TRÊN CƠ SYNTHETIC DATA: Loại bỏ các biến thời tiết hiện tại (không lag)
synthetic_leakage_cols = ['temperature_celsius', 'precipitation_mm', 'pm25_ugm3']

# Gom tất cả các cột cần drop khỏi ma trận X
drop_cols = ['record_id'] + metadata_cols + synthetic_leakage_cols

# Tách Features (X)
features = [col for col in df_train.columns if col not in drop_cols]

X_train = df_train[features]
X_test = df_test[features]

# Tách Target (y): Mô hình học trên thang Log, nhưng đánh giá trên thang gốc 0-100
y_train_log = df_train['target_log']
y_test_log = df_test['target_log']

y_train_original = df_train['vector_disease_risk_score']
y_test_original = df_test['vector_disease_risk_score']

print(f"🔥 Số lượng đặc trưng đưa vào huấn luyện: {len(features)}")
print("Các đặc trưng tiêu biểu:", features[:5])

🔥 Số lượng đặc trưng đưa vào huấn luyện: 39
Các đặc trưng tiêu biểu: ['country_code', 'year', 'month', 'week', 'latitude']


In [7]:
def calculate_metrics(y_true_original, y_pred_log):
    """
    Hàm nghịch đảo log của y_pred và tính toán các chỉ số trên thang gốc 0-100
    """
    # Inverse transform về thang gốc
    y_pred_original = np.expm1(y_pred_log)
    
    # Clip các giá trị âm hoặc vượt quá 100 nếu có do sai số mô hình
    y_pred_original = np.clip(y_pred_original, 0, 100)
    
    # Tính toán các chỉ số chuyên ngành
    mae = mean_absolute_error(y_true_original, y_pred_original)
    rmse = np.sqrt(mean_squared_error(y_true_original, y_pred_original))
    r2 = r2_score(y_true_original, y_pred_original)
    
    return {"MAE": round(mae, 4), "RMSE": round(rmse, 4), "R2": round(r2, 4)}

In [8]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.model_selection import TimeSeriesSplit

# =====================================================================
# LỚP PHÒNG THỦ: Đảm bảo ma trận đặc trưng CHỈ chứa các cột số học
# Loại bỏ triệt để các cột text/string còn sót lại (Sửa lỗi 'ARG')
# =====================================================================
X_train_fit = X_train.select_dtypes(include=[np.number])
X_test_fit = X_test.select_dtypes(include=[np.number])

print(f"✔ Ma trận huấn luyện sạch: {X_train_fit.shape[1]} đặc trưng số học.")

# 1. Khởi tạo danh sách các ứng viên baseline (Bao gồm cả GBDT Stack)
models = {
    "Linear_Regression": LinearRegression(),
    
    "Random_Forest_Default": RandomForestRegressor(
        n_estimators=100, max_depth=10, random_state=42, n_jobs=-1
    ),
    
    "XGBoost_Default": xgb.XGBRegressor(
        n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1
    ),
    
    "LightGBM_Default": lgb.LGBMRegressor(
        n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1, verbose=-1
    ),
    
    "CatBoost_Default": CatBoostRegressor(
        iterations=100, depth=6, learning_rate=0.1, random_state=42, verbose=0
    )
}

# Khởi tạo chiến lược tách chuỗi thời gian cuộn chiếu (5 tầng)
tscv = TimeSeriesSplit(n_splits=5)
baseline_results = {}

# 2. Chạy vòng lặp huấn luyện sòng phẳng
for model_name, model in models.items():
    print(f"\n===== Huấn luyện mô hình: {model_name} =====")
    
    # A. Đánh giá qua Cross Validation chuỗi thời gian
    cv_r2_scores = []
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train_fit)):
        # Tách dữ liệu theo fold (Sử dụng ma trận số học đã lọc)
        X_tr, y_tr = X_train_fit.iloc[train_idx], y_train_log.iloc[train_idx]
        X_va, y_va_orig = X_train_fit.iloc[val_idx], y_train_original.iloc[val_idx]
        
        # Train fold model
        model.fit(X_tr, y_tr)
        
        # Predict fold
        preds_log = model.predict(X_va)
        fold_metrics = calculate_metrics(y_va_orig, preds_log)
        cv_r2_scores.append(fold_metrics["R2"])
        
    mean_cv_r2 = np.mean(cv_r2_scores)
    print(f"-> Mean Cross-Validation R²: {mean_cv_r2:.4f}")
    
    # B. Huấn luyện lại trên toàn bộ tập Train và Đánh giá trên Test Set (2023+)
    model.fit(X_train_fit, y_train_log)
    test_preds_log = model.predict(X_test_fit)
    test_metrics = calculate_metrics(y_test_original, test_preds_log)
    
    # Lưu kết quả tổng hợp
    baseline_results[model_name] = {
        "Mean_CV_R2": round(mean_cv_r2, 4),
        "Test_MAE": test_metrics["MAE"],
        "Test_RMSE": test_metrics["RMSE"],
        "Test_R2": test_metrics["R2"]
    }
    print(f"-> Kết quả trên TEST SET (2023+): MAE={test_metrics['MAE']}, RMSE={test_metrics['RMSE']}, R²={test_metrics['R2']}")

# =====================================================================
# 3. Xuất bảng tổng hợp tường minh kết quả các mô hình
# =====================================================================
df_baseline_report = pd.DataFrame(baseline_results).T
print("\n" + "="*20 + " BẢNG XẾP HẠNG BASELINE " + "="*20)
print(df_baseline_report.sort_values(by="Test_R2", ascending=False))

✔ Ma trận huấn luyện sạch: 38 đặc trưng số học.

===== Huấn luyện mô hình: Linear_Regression =====
-> Mean Cross-Validation R²: 0.1252
-> Kết quả trên TEST SET (2023+): MAE=7.4298, RMSE=12.7392, R²=0.4669

===== Huấn luyện mô hình: Random_Forest_Default =====
-> Mean Cross-Validation R²: 0.7678
-> Kết quả trên TEST SET (2023+): MAE=4.3239, RMSE=7.2919, R²=0.8253

===== Huấn luyện mô hình: XGBoost_Default =====
-> Mean Cross-Validation R²: 0.6825
-> Kết quả trên TEST SET (2023+): MAE=4.363, RMSE=7.3882, R²=0.8207

===== Huấn luyện mô hình: LightGBM_Default =====
-> Mean Cross-Validation R²: 0.7750
-> Kết quả trên TEST SET (2023+): MAE=4.3025, RMSE=7.2088, R²=0.8293

===== Huấn luyện mô hình: CatBoost_Default =====
-> Mean Cross-Validation R²: 0.6515
-> Kết quả trên TEST SET (2023+): MAE=4.3848, RMSE=7.4173, R²=0.8193

==================== BẢNG XẾP HẠNG BASELINE ====================
                       Mean_CV_R2  Test_MAE  Test_RMSE  Test_R2
LightGBM_Default           0.7750    4.302

In [9]:
# Biến đổi kết quả thành DataFrame để nhìn tổng quan
df_comparison = pd.DataFrame(baseline_results).T
print("\n=== BẢNG SO SÁNH HIỆU NĂNG BASELINE MÔ HÌNH ===")
print(df_comparison)

# Tạo thư mục lưu log nếu chưa có
os.makedirs('../metrics', exist_ok=True)

# Ghi kết quả ra file JSON
with open('../metrics/baseline_results.json', 'w') as f:
    json.dump(baseline_results, f, indent=4)

print("\n-> Đã lưu log kết quả tại '../metrics/baseline_results.json'")


=== BẢNG SO SÁNH HIỆU NĂNG BASELINE MÔ HÌNH ===
                       Mean_CV_R2  Test_MAE  Test_RMSE  Test_R2
Linear_Regression          0.1252    7.4298    12.7392   0.4669
Random_Forest_Default      0.7678    4.3239     7.2919   0.8253
XGBoost_Default            0.6825    4.3630     7.3882   0.8207
LightGBM_Default           0.7750    4.3025     7.2088   0.8293
CatBoost_Default           0.6515    4.3848     7.4173   0.8193

-> Đã lưu log kết quả tại '../metrics/baseline_results.json'


In [9]:
print(df_test["vector_disease_risk_score"].describe())

count    3675.000000
mean       12.466231
std        17.449353
min         0.000000
25%         3.500000
50%         5.800000
75%         9.100000
max       100.000000
Name: vector_disease_risk_score, dtype: float64
